# Error-mitigated photonic QCBM: a CPU walkthrough

A circuit Born machine changes optical phases to fit a target distribution.
The paper uses fewer-click events to estimate the desired collision-free
distribution instead of discarding all of them. We compare post-selection
and recycling at the same **initial-shot** budget.

Reference: Salavrakos et al., *Physical Review A* **111**, L030401 (2025),
[DOI](https://doi.org/10.1103/PhysRevA.111.L030401).
[Author code, pinned revision](https://github.com/Quandela/photonic-qcbm/tree/d593d209128e403644b9aee595c474ce551391ae).
See [README](README.md), [data provenance](DATA_SOURCES.md) and
[attribution and permission limits](README.md#reference-and-attribution).

**Run All is a bounded offline demonstration:** CPU float64, a six-mode,
three-photon example, **100 SPSA updates and all five seeds 0..4** for three
conditions. It consumes **6,392,000 initial shots** including calibration and
reporting. It does not submit QPU jobs, authenticate to the Hub, download
financial data, contact a server, or start a paper-scale sweep.
Allow a few minutes on CPU. Historical Figure 2 results are loaded
separately from curated files. The original two-update smoke is still
available as `configs/toy_smoke.json`.

Use the Python environment installed from `requirements.txt` as the kernel.
Start in this paper directory or the repository root. Public cell outputs
are intentionally cleared; Run All generates fresh outputs and writes new
artifacts under ignored `outdir/` without changing committed results.

In [ ]:
import importlib.metadata as metadata
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

# Resolve both the local flat layout and a catalogue's papers/<paper>/ layout.
start = Path.cwd().resolve()
ancestors = (start, *start.parents)
root = next(path for path in ancestors if (path / "implementation.py").is_file())
candidates = (start, root / "error-qcbm", root / "papers/error_qcbm",
              root / "papers/error-qcbm")
project = next(path for path in candidates
               if (path / "configs/toy_smoke.json").is_file()
               and (path / "lib/backend.py").is_file())
sys.path[:0] = [str(project), str(root)]

import matplotlib.pyplot as plt
import merlin
import numpy as np
import perceval as pcvl
import torch
from IPython import get_ipython
from IPython.display import Markdown, display

get_ipython().run_line_magic("matplotlib", "inline")

from lib.backend import DEMO_INPUT_STATE, PhotonicBackend
from lib.backend import build_clements_circuit, initialize_parameters
from lib.data import collision_free_states, gaussian_target
from lib.spsa import distribution_loss
from lib.recycling import recycling_mitigation
from lib.training import conditional_distribution

torch.set_num_threads(1)
versions = {name: metadata.version(name) for name in
            ("merlinquantum", "perceval-quandela", "torch", "numpy", "scipy")}
print(json.dumps({"packages": versions, "merlin_module_version": merlin.__version__,
                  "device": "cpu", "dtype": "float64", "torch_threads": 1}, indent=2))

## Target, fixed output order and the paper-scale circuit

The Gaussian experiment fits a known **495-bin probability vector**, not
a finite set of training samples. The two densities are evaluated on
integer coordinates, with means 141 and 353 and widths 61, then normalized.
Their fixed bin order follows the author's binary Fock-state ordering.

A single 12-mode Clements block has 66 MZIs and **132 nominal phase
parameters**. We use the public demo input below, which differs from the
input drawn in the paper. The optical evolution includes the full
1,365-state four-photon Fock space before threshold detection. Only the
final target vector is restricted to the 495 four-click patterns.

In [ ]:
paper_states = collision_free_states(12, 4)
paper_target = gaussian_target(len(paper_states))
paper_circuit = build_clements_circuit(12)
assert len(paper_states) == 495
assert len(paper_circuit.get_parameters()) == 132
assert np.isclose(paper_target.sum(), 1.0)
print(f"Demo input: {DEMO_INPUT_STATE}")
print(f"Target bins: {len(paper_states)}; nominal phases: 132")
print(f"First bin: {paper_states[0]}; last bin: {paper_states[-1]}")
fig, ax = plt.subplots(figsize=(8, 2.7))
ax.plot(np.arange(495), paper_target)
ax.set(xlabel="Fixed output-bin index", ylabel="Target probability",
       title="Paper-scale Gaussian mixture target (no training)")
fig.tight_layout()
plt.show()

This is the actual circuit built by `lib.backend`, rendered by Perceval.
MerLin performs the optical calculations and sampling. The drawing expands
the MZI structure; it is not a conceptual replacement diagram.

In [ ]:
drawing = pcvl.pdisplay(paper_circuit, output_format=pcvl.Format.HTML,
                       recursive=True, render_size=0.65)
display(drawing)

## Small circuit: theory and native MerLin sampling

For the executable example we reduce to six modes, three photons and
30 phases. Its target has 20 bins, means 6 and 14, and widths 2.5.
This is an educational finite-shot comparison, **not** the paper's Gaussian run.

Here transmission is $T=0.5$ (loss probability $0.5$) and overlap is $I=1$.
With only 2,000 initial shots, relatively few three-click events survive.
This makes the information discarded by post-selection visible. The
setting was chosen after exploratory two-photon and three-photon pilots;
it is an illustration, not a preregistered performance benchmark. More
updates in the original two-photon toy alone did not give a consistent
recycling advantage. Three photons allow both recycled sectors to
contain non-vacuum click patterns, closer to the paper's four-photon setup.
Brightness remains one, since $T$ already includes the total efficiency.
The stock MerLin sampler is used (`native_count_accumulation="python"`).
Vacuum and all lower-click sectors remain in the initial-shot count ledger.
A two-photon collision can also produce one threshold click; a lower click
count therefore does not identify photon loss by itself.

In [ ]:
toy = json.loads((project / "configs/toy_comparison.json").read_text(encoding="utf-8"))
input_state = toy["model"]["input_state"]
states = collision_free_states(len(input_state), sum(input_state))
target = gaussian_target(len(states), toy["dataset"]["means"],
                         toy["dataset"]["standard_deviations"])
noisy_case = next(case for case in toy["training"]["conditions"]
                  if case["name"] == "postselection")
profile = {"name": "walkthrough_loss", "transmittance": noisy_case["transmittance"],
           "indistinguishability": 1.0, "g2": 0.0, "detector": "threshold"}
backend = PhotonicBackend(input_state, profile, dtype=torch.float64,
                          device="cpu", native_count_accumulation="python")
initial = initialize_parameters(len(backend.parameter_names), [0])[0]
backend.set_parameters(initial)
theory = backend.exact()
shots = 2000
counts = backend.sample(shots, seed=10000)
assert sum(counts.values()) == shots
assert np.isclose(sum(theory.values()), 1.0)
assert all(count >= 0 for count in counts.values())

sectors = []
for clicks in range(sum(input_state) + 1):
    expected = sum(prob for state, prob in theory.items() if sum(state) == clicks)
    observed = sum(count for state, count in counts.items() if sum(state) == clicks)
    sectors.append((clicks, expected, observed, observed / shots))
display(Markdown("| Clicks | Theoretical mass | Native counts | Observed fraction |\n"
                 "|---:|---:|---:|---:|\n" + "\n".join(
                     f"| {k} | {p:.6f} | {n} | {q:.6f} |"
                     for k, p, n, q in sectors)))
print(f"Initial attempts: {shots}; retained three-click events: {sectors[-1][2]}")

## Post-selection and recycling from the same count ledger

Post-selection renormalizes only the three-click sector. Recycling uses the
official Perceval photon_recycling() function through a thin adapter. It combines
the zero-, one- and two-missing-click sectors and performs the SDK's
constrained extrapolation, clipping and normalization. Missing clicks
include threshold collisions. Empty required sectors raise an error; no
extra shots are drawn to repair them.

Both estimates below use **the same 2,000 initial shots**. The reported
objective is $\mathrm{KL}(p\Vert q)$ with logarithm base two. A single small
sample does not guarantee that recycling is closer to the target.

This is an untrained random circuit: distance to the training target at
this point is not an estimator-accuracy test. The repeated training
comparison below is the relevant learning demonstration.

The adapter validates integer counts, converts them to Perceval BSCount,
and aligns the returned probabilities with the fixed target bins. Perceval
performs the sector expansion, fitting, clipping and normalization.

In [ ]:
postselected = conditional_distribution(counts, states)
recycled = recycling_mitigation(counts, len(input_state), sum(input_state))
assert recycled["states"] == states
estimates = {"Post-selection": postselected,
             "Recycling": recycled["probabilities"]}
for label, probabilities in estimates.items():
    assert np.isfinite(probabilities).all() and (probabilities >= 0).all()
    assert np.isclose(probabilities.sum(), 1.0)
    print(f"{label}: KL(model || target) = "
          f"{distribution_loss(probabilities, target, 'kl'):.6f}")
print("Recycling sector counts:", recycled["sector_counts"])
fig, ax = plt.subplots(figsize=(8, 3))
x = np.arange(len(states))
ax.bar(x - 0.175, postselected, 0.35, label="Post-selection")
ax.bar(x + 0.175, recycled["probabilities"], 0.35, label="Recycling")
ax.plot(x, target, "ko--", label="Toy target")
ax.set(xlabel="Fixed output-bin index", ylabel="Conditional probability",
       title="Same phases and count ledger; toy scope")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## A repeated finite-shot learning comparison through the shared CLI

We run `configs/toy_comparison.json`: **100 updates, five initializations
(0..4), and three conditions**. The circuit, target, 2,000 initial shots per
call, initial phases and perturbation seeds are matched across conditions.
Both noisy conditions have transmission 0.5; lossless has transmission one.
Gain calibration follows the author's same rule in every condition.

The original two-update smoke only checks wiring. Here we plot every seed,
the mean and standard deviation, and the paired final objectives. We do not
choose the best training seed or assert that recycling must win.

For this 30-parameter toy, the author's absolute-mean calibration is undefined
when the first Bernoulli perturbation has zero sum. Offset 100000 is the first
offset at or above 100000 with nonzero sums for all five seeds, checked without
evaluating a loss. The same offset is used for every condition. Earlier
12-parameter pilots failed for seeds 3 and 4 at this offset; changing the
teaching circuit does not change the optimizer or hide those exploratory failures.

Each run uses 1 initial + 2 calibration + 200 perturbation + 10 reporting calls
= **213 calls**. Fifteen runs use **6,390,000 initial shots**. The preceding
fixed-circuit example adds 2,000. These are exploratory teaching settings;
the independently recorded 20-seed paper-scale results appear separately below.

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
walkthrough = project / "outdir" / f"notebook_{stamp}"
walkthrough.mkdir(parents=True, exist_ok=False)
command = [sys.executable, str(root / "implementation.py"),
           "--paper", project.name, "--config", "configs/toy_comparison.json",
           "--outdir", str(walkthrough / "training"),
           "--data-root", str(walkthrough / "data"),
           "--device", "cpu", "--dtype", "float64", "--log-level", "warning"]
execution = subprocess.run(command, cwd=root, capture_output=True, text=True,
                           encoding="utf-8", errors="replace", timeout=600)
(walkthrough / "cli.log").write_text(execution.stdout + execution.stderr,
                                     encoding="utf-8")
if execution.returncode:
    raise RuntimeError("Toy CLI failed; inspect cli.log in the new walkthrough directory")
runs = sorted((walkthrough / "training").glob("run_*"))
assert len(runs) == 1
run_dir = runs[0]
campaign = json.loads((run_dir / "campaign.json").read_text(encoding="utf-8"))
assert campaign["status"] == "complete"
expected_instances = len(toy["training"]["seeds"]) * len(toy["training"]["conditions"])
assert campaign["completed_instances"] == expected_instances
assert campaign["failed_instances"] == 0
print("Saved run:", run_dir.relative_to(project).as_posix())
print(f"Campaign complete: {expected_instances} runs, "
      f"{toy['training']['steps']} updates each, no QPU jobs")

## Compare every initialization and reload the saved results

The left panel shows all five curves faintly, with their mean and standard
deviation. The middle panel pairs final post-selection and recycling objectives
by initialization. The right panel averages the five final estimated
distributions; no seed is selected for its appearance. Values come from this
execution's `run_status.json` and `final.npz` files.

These are finite-shot **estimator-dependent training objectives**: a lower
recycling objective alone is not proof of a more accurate underlying physical
distribution. Individual runs can worsen or cross. The purpose is to make the
learning protocol and the low-count regime visible, alongside the separately
labelled paper-scale evidence.

Reporting checkpoints are saved. Durable update-boundary continuation uses the
separate resumable campaign path described in [TRAINING.md](lib/TRAINING.md).
Rerunning this notebook creates a fresh run, rather than resuming a campaign.

In [ ]:
reports = {}
final_losses = {}
total_training_shots = 0
seeds = toy["training"]["seeds"]
steps = toy["training"]["steps"]
record_every = toy["training"]["record_every"]
expected_calls = 3 + 2 * steps + len(range(record_every, steps + 1, record_every))
colors = {"lossless": "#777777", "postselection": "#d95f02", "recycling": "#1b9e77"}
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for case in toy["training"]["conditions"]:
    name = case["name"]
    histories, distributions, finals = [], [], []
    for seed in seeds:
        directory = run_dir / f"seed_{seed:02d}" / name
        report = json.loads((directory / "run_status.json").read_text(encoding="utf-8"))
        assert report["status"] == "complete" and report["completed_steps"] == steps
        assert report["objective_calls"] == expected_calls and report["qpu_jobs_submitted"] == 0
        assert report["history"][-1]["parameter_change_norm"] > 0
        total_training_shots += report["total_initial_shots"]
        with np.load(directory / "final.npz") as saved:
            assert saved["final_counts"].sum() == shots
            assert np.isfinite(saved["parameters"]).all()
            assert np.isclose(saved["sampled_probabilities"].sum(), 1.0)
            distributions.append(saved["sampled_probabilities"].copy())
        history = np.array([row["finite_shot_loss"] for row in report["history"]])
        iterations = [row["step"] for row in report["history"]]
        histories.append(history)
        finals.append(history[-1])
        reports[(seed, name)] = report
        axes[0].plot(iterations, history, color=colors[name], alpha=0.18, linewidth=0.8)
    histories = np.asarray(histories)
    mean, std = histories.mean(axis=0), histories.std(axis=0, ddof=1)
    axes[0].plot(iterations, mean, color=colors[name], linewidth=2, label=name)
    axes[0].fill_between(iterations, np.maximum(0, mean - std), mean + std, color=colors[name], alpha=0.12)
    final_losses[name] = np.asarray(finals)
    axes[2].plot(x, np.mean(distributions, axis=0), "o-", color=colors[name], label=name)
    print(f"{name}: initial mean KL={mean[0]:.4f}; "
          f"final mean KL={mean[-1]:.4f} +/- {std[-1]:.4f} (SD, n={len(seeds)})")
for index, seed in enumerate(seeds):
    pair = [final_losses["postselection"][index], final_losses["recycling"][index]]
    axes[1].plot([0, 1], pair, "o-", alpha=0.7, label=f"seed {seed}")
wins = int(np.sum(final_losses["recycling"] < final_losses["postselection"]))
print(f"Recycling lower final finite-shot objective: {wins}/{len(seeds)} matched seeds")
assert total_training_shots == expected_calls * shots * expected_instances
axes[0].set(xlabel="SPSA update", ylabel="KL (base 2)",
            title=f"All {len(seeds)} seeds; mean +/- SD")
axes[1].set(xticks=[0, 1], xticklabels=["Post-selection", "Recycling"],
            ylabel="Final finite-shot KL", title="Every paired initialization")
axes[2].plot(x, target, "k--", label="Toy target")
axes[2].set(xlabel="Fixed output-bin index", ylabel="Conditional probability",
            title="Mean final estimates; all seeds")
for ax in axes:
    ax.legend(fontsize=8)
fig.suptitle(f"Educational low-count regime: T={noisy_case['transmittance']}, I=1; "
             f"{shots:,} initial shots per call", fontsize=11)
fig.tight_layout()
fig.savefig(walkthrough / "toy_results.png", dpi=150)
plt.show()
print(f"Verified total initial shots: {shots + total_training_shots:,}")

## Historical Figure 2 counterparts: distinct from the toy run

The following statistics and figures are reconstructed from the curated
public CSV bundle, with **no new photon shots**. Gaussian and financial
campaigns each have 20 matched seeds across three conditions. The Belenos
family has three failed original attempts and three separate sampling-stream
retries. The failures are retained; primary paired comparisons exclude
retries. Full-run outputs are not fabricated from this notebook's toy run.

Recycling has a lower final sampled objective for Gaussian 20/20, financial
20/20 and Belenos 19/19 complete original pairs. These comparisons concern
the recorded finite-shot training objectives. The Gaussian figure below
shows the actual 12-mode/four-photon, 1,000-update campaign and its
20-seed mean and spread; it is the main reproduction evidence, not a
result of the small illustrative run above.

Panel (e) shows only original seeds 0, 1 and 2. Panel (f) keeps **seed 6**
across all conditions, selected by the lowest final finite-shot KL among
the 19 completed original noisy recycling runs. It does not mix a best
seed from each condition. $T=.028$, $I=.729$ represents a model informed by
Belenos characteristics, not an actual QPU experiment. $I=.84$ is an
additional model-input ablation with an unresolved measurement convention.

In [ ]:
from lib.results import load_result_bundle, make_figures, summarize_results

bundle = load_result_bundle(project / "results")
summary = summarize_results(bundle)
assert summary["attempts"] == 263
assert summary["original_failed"] == summary["separate_retries"] == 3
assert summary["panel_f"]["seed"] == 6
for experiment, treatment, baseline in (
        ("gaussian", "recycling", "postselection"),
        ("financial", "recycling", "postselection"),
        ("belenos", "belenos_recycling", "belenos_postselection")):
    comparison = summary["comparisons"][f"{experiment}/{treatment}_vs_{baseline}"]
    print(f"{experiment}: sampled wins {comparison['finite_treatment_wins']}/"
          f"{comparison['n']}")
figures = make_figures(bundle)
display(figures["gaussian_fig2ab"])
display(figures["belenos_fig2ef_original"])
for figure in figures.values():
    plt.close(figure)
print("Historical records: 263 attempts; three original failures and three separate retries")
print("No new sampling or QPU jobs in historical result rendering")

## What to run next

- [README](README.md): default checks, offline profile smoke, full named
  configurations, output/data roots and installation instructions.
- [Results and captions](results/README.md): all Figure 2 counterparts,
  finite-shot metrics, bin grouping and explicit seed-selection rules.
- [Data sources](DATA_SOURCES.md): synthetic Gaussian definition and the
  ECB financial adaptation. The paper's original financial data were not
  released. Offline preparation requires the specified frozen CSV; live
  ECB acquisition timed out during review.
- [Sampling qualification](lib/SAMPLING.md): environment-matched 50M evidence
  is required before enabling the optimized native count adapter. This
  notebook uses the stock path and does not qualify the adapter.
- [Training and continuation](lib/TRAINING.md): SPSA, shot accounting,
  checkpoints and explicit changed-stream retries.

Full experiments use 12 modes, four photons and 1,000 updates per seed,
evaluated every ten updates. Their per-call budgets are 200k (Gaussian),
60k (financial) and 50M (Belenos characteristics). Full CPU campaigns can
take many hours per seed; this notebook does not estimate a GPU speedup.
Missing g2, drift, dark counts and mode-dependent loss prevent interpreting
the Belenos model as a complete device replica. These comparisons do not
establish superiority over classical generative models.